# Symbolic Music MoE Transformer — One-Run Google Colab

This notebook is an automated launcher. After the repository is available, the pipeline handles dataset download, validation/cleaning, tokenization, sharding, training, Google Drive checkpoints, automatic resume, generation, and evaluation.

**Model:** decoder-only Transformer + LOW/MID/HIGH learned MoE routing + cross-register interaction. No LSTM.

Enable **Runtime → Change runtime type → GPU** before running.

In [ ]:
# 1. Bootstrap the repository
# Set REPO_URL to your GitHub repository once you push this project.
REPO_URL = ''  # e.g. https://github.com/YOUR_USERNAME/music-generation.git
PROJECT_DIR = '/content/music-generation'

import os, subprocess

if not os.path.exists(PROJECT_DIR):
    if not REPO_URL:
        raise RuntimeError('Set REPO_URL to your GitHub repository, then rerun this cell.')
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)

%cd /content/music-generation
!nvidia-smi

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
# 3. Mount Google Drive. Checkpoints survive Colab runtime disconnects.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/musicgen_checkpoints'
!mkdir -p "$DRIVE_ROOT"

In [ ]:
# 5. Inspect outputs
!find "$DRIVE_ROOT/data/tokenized" -maxdepth 2 -type f | head -30
!ls -lh "$DRIVE_ROOT/checkpoints"
!ls -lh generated evaluation 2>/dev/null || true

In [ ]:
# 5. Inspect outputs
!find data/tokenized -maxdepth 2 -type f | head -30
!ls -lh "$DRIVE_ROOT"
!ls -lh generated evaluation 2>/dev/null || true

## Resume after a Colab disconnect

Reconnect the GPU, rerun cells 1–4, and the pipeline detects `latest.pt` on Google Drive and continues from the saved training state. The dataset download, cleaning, and tokenization are also skipped if their outputs already exist.

Use `--force-data` only when you intentionally want to rebuild the downloaded/processed/tokenized dataset.